# 042 数据清洗：微博数据

In [28]:
TOPIC_WEIBO_PATH = r'..\data\fe\topic_weibo.parquet'
USER_WEIBO_PATH = r'..\data\fe\user_weibo.parquet'

TOPIC_COMMENT_PATH = r"..\data\cleaned\topic_comment.parquet"

In [29]:
import pandas as pd

df_topic_weibo = pd.read_parquet(TOPIC_WEIBO_PATH)
df_user_weibo = pd.read_parquet(USER_WEIBO_PATH)

df_topic_comment = pd.read_parquet(TOPIC_COMMENT_PATH)

## `df_user_weibo` 清洗

In [30]:
# ========== 1.1 去重 ==========
# user_weibo: 68,809 组同用户重复 weibo_id
# 策略：对于同一 weibo_id，保留第一条（同用户重复取其一）；
#        对于多用户同 weibo_id（82 组，转发关系），按 user_id 区分后保留
n_before = len(df_user_weibo)
df_user_weibo = df_user_weibo.drop_duplicates(subset=["weibo_id", "user_id"], keep="first")
print(f"\nuser_weibo 去重前：{n_before:,}  去重后: {len(df_user_weibo):>10,}")
print(f"  保留率: {len(df_user_weibo) / n_before * 100:.1f}%")


user_weibo 去重前：919,646  去重后:    598,278
  保留率: 65.1%


In [31]:
# 记录表原始行数，用于全程追踪保留率
_ORIGINAL_COUNTS = {
    "topic_weibo": len(df_topic_weibo),
    "user_weibo": len(df_user_weibo),
}

def retention_rate(table: str, current_df) -> str:
    """计算相对于原始数据的保留率。"""
    orig = _ORIGINAL_COUNTS[table]
    cur = len(current_df)
    return f"{cur:,} / {orig:,} ({cur / orig * 100:.1f}% 保留)"

print("\n📦 原始数据量:")
print(f"  topic_weibo:   {len(df_topic_weibo):>10,}")
print(f"  user_weibo:    {len(df_user_weibo):>10,}")


📦 原始数据量:
  topic_weibo:        4,747
  user_weibo:       598,278


In [32]:
import re

def clean_text(text: str) -> str:
    """清洗微博评论文本，保留情绪信号。

    清洗步骤（有序）：
    1. 移除 HTML 标签
    2. 移除 URL
    3. 规范化空白字符
    """
    if not isinstance(text, str) or len(text) == 0:
        return text

    # 1. 移除 HTML 标签
    text = re.sub(r'<[^>]+>', '', text)

    # 2. 移除 URL
    text = re.sub(r'https?://\S+', '', text)

    # 3. 规范化空白（多个空白合并为一个，去首尾空白）
    text = re.sub(r'\s+', ' ', text).strip()

    return text


# 应用清洗
df_topic_weibo["content"] = df_topic_weibo["content"].apply(clean_text)
df_user_weibo["content"] = df_user_weibo["content"].apply(clean_text)

In [33]:
ad_keywords = [
    "分享有礼", "随机抽奖",
    "限时特卖",
    "领取优惠券", "购买请戳",
    "粉丝福利", "转发+评论"
]

ad_keyword = "粉丝福利"

df_user_weibo[df_user_weibo["content"].str.contains('|'.join(ad_keywords))]

,weibo_id,user_id,screen_name,content,text_length,text_quality,text_quality_label,create_time,year,month,...,hour,weekday,like_count,comment_count,repost_count,engagement,is_repost,reposted_weibo_id,topics,at_users
5092,5271231995314702,1056001684,龙哥in上海,豆豆粉丝福利,6,3,可分析,2026-02-28 09:45:05,2026,2,...,9,Saturday,0,0,0,0,True,5270326048987687,,
9667,5269792001887028,5830932596,阿云嘎嘎的闪电宝,【热烈庆祝开工第一天抽奖】 初春始发，诸事顺利，阿老师和各位小羊马年开工大吉！事业兴旺，学习...,177,3,可分析,2026-02-24 10:23:04,2026,2,...,10,Tuesday,536,481,490,1507,False,-1,阿云嘎开工大吉,
12699,5265513045300125,7281240391,MagicOS,年味加载，AI 已就位！ 🎁关注@MagicOS，带#新年荣耀AI到福到##荣耀MagicO...,181,3,可分析,2026-02-12 15:00:01,2026,2,...,15,Thursday,1429,1944,2252,5625,False,-1,"新年荣耀AI到福到,荣耀MagicOS,新年荣耀马上来到",MagicOS
12723,5263387426031360,7281240391,MagicOS,【关注抽荣耀500 Pro🎁】 小年赴家宴，MagicOS邀你畅聊迎新年！ 🔥小年玩点新花样...,314,3,可分析,2026-02-06 18:13:34,2026,2,...,18,Friday,4031,7483,8866,20380,False,-1,"新年荣耀AI到福到,荣耀MagicOS,新年荣耀马上来到",MagicOS
13451,5209793063027983,8244119413,中国军工,【20万粉丝福利！独一无二的罗布泊石+纪念徽章1套】在我国第一颗原子弹爆炸成功60周年之际，...,276,3,可分析,2025-09-11 20:48:42,2025,9,...,20,Thursday,2503,2997,6085,11585,False,-1,中国军工立flag,"微博抽奖平台,中国军工"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
878423,5116151868950543,6723106540,东书房酒馆,【转发抽奖】男人为什么要喝酒？男人喝酒，很多时候不为醉，只为在七分微醺中，回到了那个，可以天...,235,3,可分析,2024-12-27 11:12:02,2024,12,...,11,Friday,152,237,5731,6120,False,-1,"东书房送给父亲的新年礼,东书房酒年终囤酒季, 东书房醉天下,东书房酒","乔志峰,漢洋映畫,俞飞,东书房酒馆,金陵郭健,阿武看世界,超级社区平台官微"
893551,5272336027882031,1288369910,浙江卫视,【#浙江卫视连发10天粉丝福利#】#元宵节#快乐！ 浙江卫视官博连续10天福利派发|Day4...,199,3,可分析,2026-03-03 10:52:07,2026,3,...,10,Tuesday,258,416,198,872,False,-1,"浙江卫视连发10天粉丝福利,元宵节,浙浙神马福利都有","微博抽奖平台,浙江卫视"
893553,5271369155609789,1288369910,浙江卫视,【#浙江卫视连发10天粉丝福利#】 浙江卫视官博连续10天福利派发|Day1：《#太平年#》...,282,3,可分析,2026-02-28 18:50:07,2026,2,...,18,Saturday,427,452,330,1209,False,-1,"浙江卫视连发10天粉丝福利,太平年,浙浙神马福利都有","电视剧太平年,华策影视,微博抽奖平台,浙江卫视"
894854,5210058015117320,7912057882,用户1614482616,//@我是兔撕机:【抽奖】直接转发评论开一个华为耳机，原视频下评论“原创就是好”或“支持原创...,67,3,可分析,2025-09-12 14:21:32,2025,9,...,14,Friday,0,0,0,0,True,5209756960555391,,我是兔撕机


In [34]:
system_patterns = [
    "此微博已被作者删除",
    "微博可见时间范围",
    "没有这条微博的查看权限",
    "账号因违反相关法律法规",
    "该微博因违反法律法规",
    "被权利方投诉侵权",
    "用户自行申请关闭", 
    "微博社区公约", 
    "暂时无法查看", 
    "账号行为异常"
]

# 平台活动关键词
activity_keywords = [
    "点开红包", "现金红包", "微博红包", "随机抽奖",
    "抽奖详情", "领取优惠券", "购买请戳", "限时特卖",
    "分享有礼", "试试你的手气", "试手气", "抽奖平台", 
    "转发评论", "转发+评论", "转发关注", "转发+关注", 
    "关注转发", "关注+转发", "转关", "转+关", "好礼", 
    "年度歌曲", "我在参与", "免费围观", "森林驿站", 
    "开放公测", "上闲鱼", "微博智搜", "微博抓马", 
    "春节AI合拍", "微博渔场", "解锁赛博年味", "年度报告", 
    "旅行青蛙中国", "微博之夜", "粉丝福利", "好运在此", 
    "运气好到爆", "嗨抢", "欧气爆棚", "抓马福", "马年接福", 
    "微博回忆", "集福袋"
]

# 签到打卡关键词
streak_keywords = [
    "连续签到", "粉打卡", "关注超话", "签到活动", "集卡", 
    "头像挂件", "SVIP", "微博会员", 
]
print("✅ 系统提示词 & 广告关键词 已定义")

✅ 系统提示词 & 广告关键词 已定义


In [35]:
df_user_weibo[df_user_weibo["text_quality"] == 3]["content"].value_counts()

content
转发微博                                                                                                                                                                                                 73970
                                                                                                                                                                                                      8255
抱歉，根据作者设置的微博可见时间范围，此微博已不可见。                                                                                                                                                                           3877
抱歉，此微博已被作者删除。查看帮助： 网页链接                                                                                                                                                                               1436
抱歉，由于作者设置，你暂时没有这条微博的查看权限哦。查看帮助： 网页链接                                                                                                                                                

In [36]:
df_user_weibo[df_user_weibo["content"].str.contains(r"集福袋", regex=True) & (df_user_weibo["text_quality"] == 3)]

,weibo_id,user_id,screen_name,content,text_length,text_quality,text_quality_label,create_time,year,month,...,hour,weekday,like_count,comment_count,repost_count,engagement,is_repost,reposted_weibo_id,topics,at_users
13864,5249327875233456,1146661944,Theonlywinegood,#2025为自己颁奖##你好2026#快来将你的年度记忆打包，来微博一起为自己颁奖！做任务集...,68,3,可分析,2025-12-29 23:05:56,2025,12,...,23,Monday,0,0,0,0,False,-1,"2025为自己颁奖,你好2026",
13868,5248604260536139,1146661944,Theonlywinegood,#2025为自己颁奖##你好2026#快来将你的年度记忆打包，来微博一起为自己颁奖！做任务集...,68,3,可分析,2025-12-27 23:10:33,2025,12,...,23,Saturday,0,0,0,0,False,-1,"2025为自己颁奖,你好2026",
13870,5247881607905600,1146661944,Theonlywinegood,#2025为自己颁奖##你好2026#快来将你的年度记忆打包，来微博一起为自己颁奖！做任务集...,68,3,可分析,2025-12-25 23:18:58,2025,12,...,23,Thursday,0,0,0,0,False,-1,"2025为自己颁奖,你好2026",
13888,5247514360156043,1146661944,Theonlywinegood,#2025为自己颁奖##你好2026#快来将你的年度记忆打包，来微博一起为自己颁奖！做任务集...,68,3,可分析,2025-12-24 22:59:40,2025,12,...,22,Wednesday,0,0,0,0,False,-1,"2025为自己颁奖,你好2026",
125648,5248498318184977,1826720760,虎林张英,#2025为自己颁奖##你好2026#快来将你的年度记忆打包，来微博一起为自己颁奖！做任务集...,68,3,可分析,2025-12-27 16:09:34,2025,12,...,16,Saturday,2,4,0,6,False,-1,"2025为自己颁奖,你好2026",
163180,5247134115831192,1957472307,SamByRuby,#2025为自己颁奖##你好2026#快来将你的年度记忆打包，来微博一起为自己颁奖！做任务集...,68,3,可分析,2025-12-23 21:48:43,2025,12,...,21,Tuesday,0,0,0,0,False,-1,"2025为自己颁奖,你好2026",
211996,5249160397457396,2303525423,春之翔,#2025为自己颁奖##你好2026#快来将你的年度记忆打包，来微博一起为自己颁奖！做任务集...,68,3,可分析,2025-12-29 12:00:26,2025,12,...,12,Monday,0,0,0,0,False,-1,"2025为自己颁奖,你好2026",
284158,5250068762329820,2927148231,yychaha,#2025为自己颁奖##你好2026#快来将你的年度记忆打包，来微博一起为自己颁奖！做任务集...,68,3,可分析,2026-01-01 00:09:56,2026,1,...,0,Thursday,0,0,0,0,False,-1,"2025为自己颁奖,你好2026",
397668,5249935094056202,5342875988,Remember丶菜,#2025为自己颁奖##你好2026#快来将你的年度记忆打包，来微博一起为自己颁奖！做任务集...,68,3,可分析,2025-12-31 15:18:48,2025,12,...,15,Wednesday,0,0,0,0,False,-1,"2025为自己颁奖,你好2026",
456162,5250069882472791,5683918769,月冷迟迟,#2025为自己颁奖##你好2026#快来将你的年度记忆打包，来微博一起为自己颁奖！做任务集...,68,3,可分析,2026-01-01 00:14:24,2026,1,...,0,Thursday,0,0,0,0,False,-1,"2025为自己颁奖,你好2026",


In [37]:
# ========== 3. 文本质量分级机制 ==========
# 本阶段为 df_user_weibo 添加文本质量等级字段，用于区分微博文本分析价值
# 
# 级别定义：
# - Level 0 "系统提示" / "空内容"：无有效用户表达
#   - "系统提示"：平台生成的系统提示（删帖、权限等）
#   - "空内容"：清洗后为空的微博文本
# - Level 1 "广告、抽奖、营销等"：虽为用户文本，但主要用于商业推广、活动参与，模板化程度高
# - Level 2 "低信息量"：属于用户文本，但内容极少，信息密度极低（纯数字/符号/字母/Emoji）
# - Level 3 "普通内容"：具有基本语义内容的普通微博，可供后续情绪分析使用
# - Level 4 "高质量内容"：当前不需要识别（暂留作未来扩展）

# ========== 3.1 初始化质量等级字段 ==========
df_user_weibo["text_quality"] = 3  # 默认设为 Level 3（普通内容）
df_user_weibo["text_quality_label"] = "普通内容"

print(f"✅ user_weibo 初始化质量等级字段")
print(f"   Shape: {df_user_weibo.shape}")

# ========== 3.2 定义质量判定函数 ==========

# 规则集：完全匹配的正则表达式（按优先级排序）
QUALITY_RULES = [
    # Level 1: 占位符和功能互动
    (r'^([转轉][发發]?(至?微博( 查看图片)?)?|Repost|分享(图片|新鲜事|视频)|网页链接|[存码马](克)?|转一个|签到|收藏)$', 1, '占位/功能互动'),
    (r'^#[^#]+#(\s*#[^#]+#)*$', 1, '纯话题占位'),
    
    # Level 2: 参与互动和祝福
    (r'^(我?来[了啦]{0,2}|接{1,3}|[抽中]|了解一下|收到|是的|关注|期待)$', 2, '参与互动'),
    (r'^((新年|元宵节|生日|除夕)快乐[！!]?|开工大吉|[早晚]安|早上好)$', 2, '固定祝福语'),
    (r'^(好(的|好好)?|哈{2,}|哇(哦)?|嗯)$', 2, '纯感叹'),
    (r'[\U0001F300-\U0001F9FF\U00002600-\U000027BF'
     r'\U0001FA00-\U0001FA6F\U0001FA70-\U0001FAFF\uFE00-\uFE0F\u200D]+', 2, '纯表情')

]

def classify_text_quality(text: str) -> tuple:
    """根据微博文本内容，判定其质量等级。
    
    Args:
        text (str): 微博文本内容
    
    Returns:
        tuple: (quality_level: int, quality_label: str)
    """
    
    # 先检查是否为空内容（清洗后为空字符串）
    if not isinstance(text, str) or len(text.strip()) == 0:
        return (0, "空内容")
    
    t = text.strip()
    
    # Level 0：系统提示词（平台生成，无有效用户表达）
    for pattern in system_patterns:
        if pattern in t:
            return (0, "系统提示")
    
    # Level 1：广告、抽奖、营销等（模板化内容，商业导向）
    # 平台活动模板
    for keyword in activity_keywords:
        if keyword in t:
            return (1, "平台活动模板")
    # 签到打卡模板
    for keyword in streak_keywords:
        if keyword in t:
            return (1, "签到打卡模板")
    
    # ========== 新增：完全匹配的规则（按优先级） ==========
    for pattern, level, label in QUALITY_RULES:
        if re.fullmatch(pattern, t):
            return (level, label)

    # Level 2：低信息量文本（用户文本，但内容极少）
    if _is_low_info_text(t):
        return (2, "低信息量")
    
    # Level 3：普通内容（默认，具有基本语义内容）
    return (3, "普通内容")


def _is_low_info_text(text: str) -> bool:
    """判断文本是否为低信息量（纯数字/纯符号/纯英文字母/纯Emoji）。
    
    Args:
        text (str): 已去空格的文本
    
    Returns:
        bool: True 表示低信息量，False 表示有信息量
    """
    # 纯数字
    if re.fullmatch(r'\d+', text):
        return True
    # 纯符号（不含字母、数字、汉字、Emoji）
    if re.fullmatch(r'[^\w\u4e00-\u9fff\U00010000-\U0010FFFF]+', text):
        return True
    # 纯英文字母
    if re.fullmatch(r'[a-zA-Z]+', text):
        return True
    return False


# ========== 3.4 应用质量分级 ==========
print("\n\n📊 应用文本质量分级...")
df_user_weibo[["text_quality", "text_quality_label"]] = df_user_weibo["content"].apply(
    lambda x: pd.Series(classify_text_quality(x))
)

# 统计各质量等级的数量
quality_counts = df_user_weibo["text_quality_label"].value_counts()
print("\n📈 文本质量等级分布：")
for quality_idx in range(5):
    quality_labels = {
        0: "空内容 / 系统提示",
        1: "广告、抽奖、营销等",
        2: "低信息量 / 参与互动 / 祝福语 / 表情",
        3: "普通内容",
        4: "高质量内容"
    }
    label = quality_labels[quality_idx]
    count = len(df_user_weibo[df_user_weibo["text_quality"] == quality_idx])
    if count > 0:
        pct = count / len(df_user_weibo) * 100
        # 对于 Level 0，显示细分信息
        if quality_idx == 0:
            empty_count = len(df_user_weibo[df_user_weibo["text_quality_label"] == "空内容"])
            system_count = len(df_user_weibo[df_user_weibo["text_quality_label"] == "系统提示"])
            print(f"  Level {quality_idx} ({label:20s}): {count:>10,} ({pct:5.2f}%)")
            print(f"       ├─ 空内容: {empty_count:>10,}")
            print(f"       └─ 系统提示: {system_count:>10,}")
        else:
            print(f"  Level {quality_idx} ({label:20s}): {count:>10,} ({pct:5.2f}%)")

# 显示新增规则的分布
print("\n📊 Level 1 标签分布（新增规则）:")
level1_labels = df_user_weibo[df_user_weibo["text_quality"] == 1]["text_quality_label"].value_counts()
for label, count in level1_labels.items():
    pct = count / len(df_user_weibo[df_user_weibo["text_quality"] == 1]) * 100
    print(f"  {label:20s}: {count:>10,} ({pct:5.2f}%)")

print("\n📊 Level 2 标签分布（新增规则）:")
level2_labels = df_user_weibo[df_user_weibo["text_quality"] == 2]["text_quality_label"].value_counts()
for label, count in level2_labels.items():
    pct = count / len(df_user_weibo[df_user_weibo["text_quality"] == 2]) * 100
    print(f"  {label:20s}: {count:>10,} ({pct:5.2f}%)")

print(f"\n✅ 文本质量分级完成: {retention_rate('user_weibo', df_user_weibo)}")


✅ user_weibo 初始化质量等级字段
   Shape: (598278, 21)


📊 应用文本质量分级...

📈 文本质量等级分布：
  Level 0 (空内容 / 系统提示          ):     15,962 ( 2.67%)
       ├─ 空内容:      8,255
       └─ 系统提示:      7,707
  Level 1 (广告、抽奖、营销等           ):    109,798 (18.35%)
  Level 2 (低信息量 / 参与互动 / 祝福语 / 表情):      4,917 ( 0.82%)
  Level 3 (普通内容                ):    467,601 (78.16%)

📊 Level 1 标签分布（新增规则）:
  占位/功能互动             :     77,041 (70.17%)
  平台活动模板              :     24,621 (22.42%)

📈 文本质量等级分布：
  Level 0 (空内容 / 系统提示          ):     15,962 ( 2.67%)
       ├─ 空内容:      8,255
       └─ 系统提示:      7,707
  Level 1 (广告、抽奖、营销等           ):    109,798 (18.35%)
  Level 2 (低信息量 / 参与互动 / 祝福语 / 表情):      4,917 ( 0.82%)
  Level 3 (普通内容                ):    467,601 (78.16%)

📊 Level 1 标签分布（新增规则）:
  占位/功能互动             :     77,041 (70.17%)
  平台活动模板              :     24,621 (22.42%)
  纯话题占位               :      6,697 ( 6.10%)
  签到打卡模板              :      1,439 ( 1.31%)

📊 Level 2 标签分布（新增规则）:
  低信息量                :      1,504 (3

## `df_topic_weibo` 清洗

In [42]:
df_topic_weibo = pd.read_parquet(TOPIC_WEIBO_PATH)


# ========== 4. 评论聚合统计并回填到 topic_weibo ==========
print("\n\n📊 开始评论聚合统计...")

# 4.1 按 weibo_id 聚合评论数据
comment_agg = df_topic_comment.groupby("weibo_id").agg(
    comment_crawled_count=("weibo_id", "size"),  # 爬取到的评论总数
    comment_hq_count=("text_quality", lambda x: (x >= 3).sum()),  # 高质量评论数
    comment_hq_user_count=("user_id", lambda x: x[df_topic_comment.loc[x.index, "text_quality"] >= 3].nunique())  # 高质量评论的去重用户数
).reset_index()

# 4.2 计算高质量评论比例
comment_agg["comment_hq_ratio"] = (comment_agg["comment_hq_count"] / comment_agg["comment_crawled_count"]).round(2)

print(f"\n✅ 评论聚合统计完成：{len(comment_agg):,} 个微博")
print(f"   字段：{list(comment_agg.columns)}")

# 4.3 将聚合结果回填到 df_topic_weibo
df_topic_weibo = df_topic_weibo.merge(
    comment_agg[["weibo_id", "comment_crawled_count", "comment_hq_count", "comment_hq_ratio", "comment_hq_user_count"]],
    on="weibo_id",
    how="left"
)

# 处理没有评论的微博（填充为 0 或相关值）
df_topic_weibo["comment_crawled_count"] = df_topic_weibo["comment_crawled_count"].fillna(0).astype(int)
df_topic_weibo["comment_hq_count"] = df_topic_weibo["comment_hq_count"].fillna(0).astype(int)
df_topic_weibo["comment_hq_ratio"] = df_topic_weibo["comment_hq_ratio"].fillna(0.0).round(2)
df_topic_weibo["comment_hq_user_count"] = df_topic_weibo["comment_hq_user_count"].fillna(0).astype(int)

print(f"\n✅ 评论统计已回填到 df_topic_weibo")

# ========== 4.4 计算话题价值等级 ==========
print("\n📊 开始计算话题价值等级...")

def calculate_topic_value(row):
    """根据评论统计信息，判定话题的价值等级。
    
    规则：
    - comment_crawled_count < 15: Level 0 "评论样本不足"
    - comment_crawled_count >= 15 且 comment_hq_count < 17: Level 1 "低价值"
    - comment_crawled_count >= 20 且 comment_hq_count >= 27 且 comment_hq_user_count >= 24: Level 3 "高价值"
    - 其余: Level 2 "一般价值"
    
    Args:
        row: DataFrame 的一行
    
    Returns:
        tuple: (topic_value: int, topic_value_label: str)
    """
    crawled = row["comment_crawled_count"]
    hq_count = row["comment_hq_count"]
    hq_user = row["comment_hq_user_count"]
    trending_type = row["trending_type"]
    
    if crawled < 15:
        return (0, "评论样本不足")
    elif crawled >= 15 and hq_count < 17:
        return (1, "低价值")
    elif trending_type is not None and crawled >= 20 and hq_count >= 27 and hq_user >= 24:
        return (3, "高价值")
    else:
        return (2, "一般价值")

# 应用话题价值计算
df_topic_weibo[["topic_value", "topic_value_label"]] = df_topic_weibo.apply(
    lambda row: pd.Series(calculate_topic_value(row)),
    axis=1
)

# 统计话题价值分布
print(f"\n📈 话题价值等级分布：")
value_counts = df_topic_weibo["topic_value_label"].value_counts().sort_index()
for label in ["评论样本不足", "低价值", "一般价值", "高价值"]:
    count = len(df_topic_weibo[df_topic_weibo["topic_value_label"] == label])
    if count > 0:
        pct = count / len(df_topic_weibo) * 100
        print(f"  {label:15s}: {count:>10,} ({pct:5.2f}%)")

print(f"\n✅ 话题价值等级计算完成")

# 4.5 调整列顺序
topic_weibo_cols = [
    # ID & 用户
    "weibo_id", "user_id", "screen_name", "gender",
    # 话题
    "topic",
    # 文本
    "content", "text_length",
    # 时间
    "create_time", "year", "month", "day", "hour", "weekday",
    # 互动
    "like_count", "comment_count", "repost_count", "engagement",
    # 爬取评论信息
    "comment_crawled_count", "comment_hq_count", "comment_hq_ratio", "comment_hq_user_count",
    # 话题价值评估
    "topic_value", "topic_value_label",
    # 热搜词条信息
    "trending_date", "trending_type", "trending_click"
]

df_topic_weibo = df_topic_weibo[topic_weibo_cols]

print(f"\n✅ 列顺序已调整")
print(f"   新的 df_topic_weibo shape: {df_topic_weibo.shape}")
print(f"   新的 df_topic_weibo columns: {list(df_topic_weibo.columns)}")



📊 开始评论聚合统计...

✅ 评论聚合统计完成：4,704 个微博
   字段：['weibo_id', 'comment_crawled_count', 'comment_hq_count', 'comment_hq_user_count', 'comment_hq_ratio']

✅ 评论统计已回填到 df_topic_weibo

📊 开始计算话题价值等级...

✅ 评论聚合统计完成：4,704 个微博
   字段：['weibo_id', 'comment_crawled_count', 'comment_hq_count', 'comment_hq_user_count', 'comment_hq_ratio']

✅ 评论统计已回填到 df_topic_weibo

📊 开始计算话题价值等级...

📈 话题价值等级分布：
  评论样本不足         :        282 ( 5.94%)
  低价值            :        798 (16.81%)
  一般价值           :      2,743 (57.78%)
  高价值            :        924 (19.46%)

✅ 话题价值等级计算完成

✅ 列顺序已调整
   新的 df_topic_weibo shape: (4747, 26)
   新的 df_topic_weibo columns: ['weibo_id', 'user_id', 'screen_name', 'gender', 'topic', 'content', 'text_length', 'create_time', 'year', 'month', 'day', 'hour', 'weekday', 'like_count', 'comment_count', 'repost_count', 'engagement', 'comment_crawled_count', 'comment_hq_count', 'comment_hq_ratio', 'comment_hq_user_count', 'topic_value', 'topic_value_label', 'trending_date', 'trending_type', 'trending

In [53]:
def sample_weibo_comments(level, n_weibo=50, n_comment=20) -> dict:
    """根据话题价值等级，从 df_topic_comment 中随机抽取微博评论样本。

    Args:
        level (int): 话题价值等级
        n_weibo (int): 抽样微博数量
        n_comment (int): 抽样评论数量

    Returns:
        dict
    """
    weibo_comment_dict = {}
    # 根据话题价值等级筛选微博
    level_weibo_count = len(df_topic_weibo[df_topic_weibo["topic_value"] == level])
    n_weibo = min(n_weibo, level_weibo_count)  # 确保抽样数量不超过可用微博数

    sampled_weibo_ids = df_topic_weibo[df_topic_weibo["topic_value"] == level].sample(n=n_weibo)["weibo_id"]
    for weibo_id in sampled_weibo_ids:
        comment_count = len(df_topic_comment[df_topic_comment["weibo_id"] == weibo_id])
        n_comment = min(n_comment, comment_count)  # 确保抽样数量不超过可用评论数
        sampled_comments = df_topic_comment[df_topic_comment["weibo_id"] == weibo_id][["screen_name", "content", "text_quality_label"]].sample(n=n_comment)
        weibo_comment_dict[weibo_id] = [(row["screen_name"], row["content"], row["text_quality_label"]) for _, row in sampled_comments.iterrows()]

    # 随机抽样
    return weibo_comment_dict

# sample_weibo_comments(3)

In [39]:
import os

# 创建输出目录
output_dir = r"..\data\cleaned"
os.makedirs(output_dir, exist_ok=True)

# 保存
datasets = {
    "topic_weibo": df_topic_weibo,
    "user_weibo": df_user_weibo,
}

for name, df in datasets.items():
    path = os.path.join(output_dir, f"{name}.parquet")
    df.to_parquet(path, index=False)
    size_mb = os.path.getsize(path) / (1024 * 1024)
    print(f"✅ {name}.parquet 已保存 ({df.shape[0]:>10,} rows × {df.shape[1]:>2} cols, {size_mb:.1f} MB)")

print(f"\n📂 输出目录: {os.path.abspath(output_dir)}")

# 显示 user_weibo 的最终字段列表
print(f"\n📋 user_weibo 最终字段列表：")
print(f"  Columns: {list(df_user_weibo.columns)}")


✅ topic_weibo.parquet 已保存 (     4,747 rows × 26 cols, 2.1 MB)
✅ user_weibo.parquet 已保存 (   598,278 rows × 21 cols, 135.4 MB)

📂 输出目录: d:\GraduationProject\data\cleaned

📋 user_weibo 最终字段列表：
  Columns: ['weibo_id', 'user_id', 'screen_name', 'content', 'text_length', 'text_quality', 'text_quality_label', 'create_time', 'year', 'month', 'day', 'hour', 'weekday', 'like_count', 'comment_count', 'repost_count', 'engagement', 'is_repost', 'reposted_weibo_id', 'topics', 'at_users']
✅ user_weibo.parquet 已保存 (   598,278 rows × 21 cols, 135.4 MB)

📂 输出目录: d:\GraduationProject\data\cleaned

📋 user_weibo 最终字段列表：
  Columns: ['weibo_id', 'user_id', 'screen_name', 'content', 'text_length', 'text_quality', 'text_quality_label', 'create_time', 'year', 'month', 'day', 'hour', 'weekday', 'like_count', 'comment_count', 'repost_count', 'engagement', 'is_repost', 'reposted_weibo_id', 'topics', 'at_users']
